# Basis Trading Tutorial: ENA Perpetuals on Binance and Backpack

Tutorial for [NautilusTrader](https://nautilustrader.io/docs/) - a high-performance algorithmic trading platform.

## Learning Objectives

In this tutorial, you will learn:
1. **Basis Trading Fundamentals** - Understanding perpetual futures basis and arbitrage
2. **Multi-Exchange Integration** - Connecting to Binance and Backpack simultaneously
3. **NautilusTrader Components** - Using MessageBus, Cache, DataEngine for live data
4. **Real-Time Data Processing** - Fetching and normalizing data from different exchanges
5. **Spread Calculation** - Computing basis between instruments with different quote currencies
6. **Strategy Development** - Building a simple basis trading strategy

## Prerequisites

- Python 3.11+ installed
- NautilusTrader installed (`pip install -U nautilus_trader`)
- API keys for Binance Futures and Backpack Exchange
- Basic understanding of futures trading

## Part 1: Introduction to Basis Trading

### What is Basis Trading?

Basis trading is a market-neutral strategy that profits from the price difference (basis) between related instruments. In crypto markets, common basis trades include:

1. **Spot vs Futures**: Trading the difference between spot and futures prices
2. **Cross-Exchange Arbitrage**: Exploiting price differences across exchanges
3. **Perpetual Funding Arbitrage**: Capturing funding rates in perpetual futures

### Our Example: ENA Perpetuals

We'll monitor and trade the basis between:
- **Binance**: ENA-USDT Perpetual Future
- **Backpack**: ENA-USDC Perpetual Future

### Key Concepts

- **Basis = Price_Exchange_A - Price_Exchange_B**
- **Positive Basis**: Exchange A is more expensive (potential short A, long B)
- **Negative Basis**: Exchange B is more expensive (potential long A, short B)
- **Mean Reversion**: Basis tends to revert to historical average

## Part 2: Setting Up NautilusTrader Components

Let's start by importing necessary modules and setting up core components.

In [2]:
# Core imports
import asyncio
import os
from decimal import Decimal
from typing import Optional, Dict, List
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# NautilusTrader imports
from nautilus_trader.adapters.backpack.common.constants import BACKPACK_VENUE
from nautilus_trader.adapters.backpack.config import BackpackDataClientConfig
from nautilus_trader.adapters.backpack.factories import BackpackLiveDataClientFactory
from nautilus_trader.adapters.binance import BINANCE, BinanceAccountType
from nautilus_trader.adapters.binance import BinanceDataClientConfig
from nautilus_trader.adapters.binance import BinanceLiveDataClientFactory
from nautilus_trader.cache.cache import Cache
from nautilus_trader.common.component import LiveClock, MessageBus, init_logging
from nautilus_trader.common.enums import LogLevel
from nautilus_trader.config import InstrumentProviderConfig
from nautilus_trader.data.engine import DataEngine
from nautilus_trader.model.data import QuoteTick, TradeTick, OrderBookDeltas
from nautilus_trader.model.identifiers import InstrumentId, Symbol, TraderId

print("✅ All modules imported successfully")

Matplotlib is building the font cache; this may take a moment.


✅ All modules imported successfully


### Initialize Core Components

NautilusTrader uses several core components that work together:

1. **LiveClock**: Provides timestamps for live trading
2. **MessageBus**: Routes messages between components (pub/sub pattern)
3. **Cache**: Stores market data and state in memory
4. **DataEngine**: Manages data clients and routes market data

In [3]:
# Initialize logging
init_logging(level_stdout=LogLevel.INFO)

# Create core components
clock = LiveClock()
trader_id = TraderId("BASIS-TUTORIAL-001")

# MessageBus for inter-component communication
msgbus = MessageBus(
    trader_id=trader_id,
    clock=clock,
)

# Cache for storing market data
cache = Cache()

# DataEngine for managing data flow
data_engine = DataEngine(
    msgbus=msgbus,
    cache=cache,
    clock=clock,
)

print("✅ Core components initialized")
print(f"  - Trader ID: {trader_id}")
print(f"  - Clock: {clock.__class__.__name__}")
print(f"  - MessageBus: Ready")
print(f"  - Cache: Ready")
print(f"  - DataEngine: Ready")

2025-08-08T21:25:15.825291000Z [INFO] TRADER-000.MessageBus: config.database=None
✅ Core components initialized
  - Trader ID: BASIS-TUTORIAL-001
  - Clock: LiveClock
  - MessageBus: Ready
  - Cache: Ready
  - DataEngine: Ready
2025-08-08T21:25:15.826097000Z [INFO] TRADER-000.MessageBus: config.encoding='msgpack'
2025-08-08T21:25:15.826100000Z [INFO] TRADER-000.MessageBus: config.timestamps_as_iso8601=False
2025-08-08T21:25:15.826101000Z [INFO] TRADER-000.MessageBus: config.buffer_interval_ms=None
2025-08-08T21:25:15.826103000Z [INFO] TRADER-000.MessageBus: config.autotrim_mins=None
2025-08-08T21:25:15.826104000Z [INFO] TRADER-000.MessageBus: config.use_trader_prefix=True
2025-08-08T21:25:15.826105000Z [INFO] TRADER-000.MessageBus: config.use_trader_id=True
2025-08-08T21:25:15.826106000Z [INFO] TRADER-000.MessageBus: config.use_instance_id=False
2025-08-08T21:25:15.826107000Z [INFO] TRADER-000.MessageBus: config.streams_prefix='stream'
2025-08-08T21:25:15.826108000Z [INFO] TRADER-000.M

## Part 3: Configuring Exchange Connections

Now we'll configure connections to both Binance and Backpack exchanges.

In [ ]:
# Check for API keys
def check_api_keys():
    """Verify API keys are configured."""
    keys_status = {
        "BINANCE_API_KEY": os.getenv("BINANCE_API_KEY") is not None,
        "BINANCE_API_SECRET": os.getenv("BINANCE_API_SECRET") is not None,
        "BACKPACK_API_KEY": os.getenv("BACKPACK_API_KEY") is not None,
        "BACKPACK_API_SECRET": os.getenv("BACKPACK_API_SECRET") is not None,
    }
    
    print("API Keys Status:")
    for key, status in keys_status.items():
        status_icon = "✅" if status else "❌"
        print(f"  {status_icon} {key}: {'Configured' if status else 'Missing'}")
    
    if not all(keys_status.values()):
        print("\n⚠️  Please set missing API keys in your .env file")
        return False
    return True

api_keys_ready = check_api_keys()

In [ ]:
# Define instrument IDs
binance_instrument_id = InstrumentId(
    Symbol("ENAUSDT"),  # Binance ENA-USDT perpetual
    BINANCE,
)

backpack_instrument_id = InstrumentId(
    Symbol("ENA_USDC_PERP"),  # Backpack ENA-USDC perpetual
    BACKPACK_VENUE,
)

print("Instruments configured:")
print(f"  - Binance: {binance_instrument_id}")
print(f"  - Backpack: {backpack_instrument_id}")

In [ ]:
# Configure Binance data client
binance_config = BinanceDataClientConfig(
    api_key=os.getenv("BINANCE_API_KEY"),
    api_secret=os.getenv("BINANCE_API_SECRET"),
    account_type=BinanceAccountType.USDT_FUTURE,  # For perpetual futures
    testnet=False,  # Use mainnet for real data
    instrument_provider=InstrumentProviderConfig(
        load_all=False,  # Only load specific instruments
    ),
)

# Configure Backpack data client
backpack_config = BackpackDataClientConfig(
    api_key=os.getenv("BACKPACK_API_KEY"),
    api_secret=os.getenv("BACKPACK_API_SECRET"),
    testnet=False,  # Use mainnet for real data
    instrument_provider=InstrumentProviderConfig(
        load_all=False,  # Only load specific instruments
    ),
)

print("✅ Exchange configurations ready")

## Part 4: Basis Calculation and Monitoring

Let's create a class to monitor and calculate the basis spread between the two perpetuals.

In [ ]:
class BasisTracker:
    """
    Tracks basis spread between two perpetual instruments.
    
    This class demonstrates:
    - Real-time quote processing
    - Spread calculation
    - Historical tracking
    - Arbitrage detection
    """
    
    def __init__(self, binance_id: InstrumentId, backpack_id: InstrumentId):
        self.binance_id = binance_id
        self.backpack_id = backpack_id
        
        # Latest quotes
        self.binance_quote: Optional[QuoteTick] = None
        self.backpack_quote: Optional[QuoteTick] = None
        
        # Historical data
        self.history: List[Dict] = []
        self.timestamps: List[datetime] = []
        
        # Statistics
        self.max_basis = None
        self.min_basis = None
        self.opportunities_detected = 0
        
    def update_binance(self, quote: QuoteTick):
        """Update Binance quote."""
        self.binance_quote = quote
        self._calculate_basis()
        
    def update_backpack(self, quote: QuoteTick):
        """Update Backpack quote."""
        self.backpack_quote = quote
        self._calculate_basis()
        
    def _calculate_basis(self):
        """Calculate current basis spread."""
        if not (self.binance_quote and self.backpack_quote):
            return
        
        # Extract prices
        binance_bid = float(self.binance_quote.bid_price)
        binance_ask = float(self.binance_quote.ask_price)
        backpack_bid = float(self.backpack_quote.bid_price)
        backpack_ask = float(self.backpack_quote.ask_price)
        
        # Calculate mid prices
        binance_mid = (binance_bid + binance_ask) / 2
        backpack_mid = (backpack_bid + backpack_ask) / 2
        
        # Calculate basis (assuming USDC ≈ USDT)
        basis = backpack_mid - binance_mid
        basis_bps = (basis / binance_mid) * 10000
        
        # Store in history
        data_point = {
            'timestamp': datetime.now(timezone.utc),
            'binance_bid': binance_bid,
            'binance_ask': binance_ask,
            'binance_mid': binance_mid,
            'backpack_bid': backpack_bid,
            'backpack_ask': backpack_ask,
            'backpack_mid': backpack_mid,
            'basis': basis,
            'basis_bps': basis_bps,
        }
        
        self.history.append(data_point)
        self.timestamps.append(data_point['timestamp'])
        
        # Update statistics
        if self.max_basis is None or basis > self.max_basis:
            self.max_basis = basis
        if self.min_basis is None or basis < self.min_basis:
            self.min_basis = basis
        
        # Check for arbitrage
        if self._check_arbitrage(binance_bid, binance_ask, backpack_bid, backpack_ask):
            self.opportunities_detected += 1
    
    def _check_arbitrage(self, b_bid, b_ask, bp_bid, bp_ask) -> bool:
        """Check for arbitrage opportunities."""
        # Buy Binance, Sell Backpack
        if bp_bid > b_ask:
            return True
        # Buy Backpack, Sell Binance
        if b_bid > bp_ask:
            return True
        return False
    
    def get_current_basis(self) -> Optional[float]:
        """Get current basis in dollars."""
        if self.history:
            return self.history[-1]['basis']
        return None
    
    def get_statistics(self) -> Dict:
        """Get trading statistics."""
        if not self.history:
            return {}
        
        basis_values = [h['basis'] for h in self.history]
        
        return {
            'current_basis': basis_values[-1],
            'min_basis': self.min_basis,
            'max_basis': self.max_basis,
            'avg_basis': sum(basis_values) / len(basis_values),
            'std_basis': pd.Series(basis_values).std(),
            'data_points': len(self.history),
            'opportunities': self.opportunities_detected,
        }
    
    def plot_basis(self):
        """Plot basis spread over time."""
        if not self.history:
            print("No data to plot")
            return
        
        df = pd.DataFrame(self.history)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        
        # Plot prices
        ax1.plot(df['timestamp'], df['binance_mid'], label='Binance Mid', color='orange')
        ax1.plot(df['timestamp'], df['backpack_mid'], label='Backpack Mid', color='blue')
        ax1.set_ylabel('Price (USD)')
        ax1.set_title('ENA Perpetual Prices')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot basis
        ax2.plot(df['timestamp'], df['basis'], label='Basis', color='green')
        ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
        ax2.fill_between(df['timestamp'], 0, df['basis'], alpha=0.3, color='green')
        ax2.set_xlabel('Time')
        ax2.set_ylabel('Basis (USD)')
        ax2.set_title('Basis Spread (Backpack - Binance)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# Create basis tracker
basis_tracker = BasisTracker(binance_instrument_id, backpack_instrument_id)
print("✅ Basis tracker initialized")

## Part 5: Fetching Live Market Data

Now let's connect to both exchanges and start receiving live market data.

In [ ]:
async def setup_data_clients():
    """
    Set up and connect data clients for both exchanges.
    
    This demonstrates:
    - Creating data client factories
    - Asynchronous client creation
    - Client registration with DataEngine
    """
    
    # Create factories
    binance_factory = BinanceLiveDataClientFactory()
    backpack_factory = BackpackLiveDataClientFactory()
    
    # Create clients
    binance_client = await binance_factory.create_async(
        loop=asyncio.get_event_loop(),
        name="BINANCE",
        config=binance_config,
        msgbus=msgbus,
        cache=cache,
        clock=clock,
    )
    
    backpack_client = await backpack_factory.create_async(
        loop=asyncio.get_event_loop(),
        name="BACKPACK",
        config=backpack_config,
        msgbus=msgbus,
        cache=cache,
        clock=clock,
    )
    
    # Register with data engine
    data_engine.register_client(binance_client)
    data_engine.register_client(backpack_client)
    
    # Start data engine
    data_engine.start()
    
    # Connect clients
    await binance_client.connect()
    await backpack_client.connect()
    
    print("✅ Data clients connected")
    return binance_client, backpack_client

# Note: This would be run in an async context
# For the notebook, we'll demonstrate the pattern
print("Data client setup pattern demonstrated")
print("In production, run: await setup_data_clients()")

## Part 6: Building a Simple Basis Trading Strategy

Let's create a simple strategy that monitors the basis and generates trading signals.

In [ ]:
from nautilus_trader.trading.strategy import Strategy
from nautilus_trader.model.enums import OrderSide, TimeInForce
from nautilus_trader.model.orders import LimitOrder
from nautilus_trader.model.objects import Price, Quantity

class BasisTradingStrategy(Strategy):
    """
    Simple basis trading strategy.
    
    Strategy logic:
    1. Monitor basis spread between Binance and Backpack
    2. Enter when basis exceeds threshold
    3. Exit when basis reverts to mean
    
    Parameters
    ----------
    binance_instrument_id : InstrumentId
        Binance perpetual instrument
    backpack_instrument_id : InstrumentId
        Backpack perpetual instrument
    basis_threshold : float
        Basis threshold in dollars to trigger entry
    trade_size : float
        Position size in base currency
    """
    
    def __init__(
        self,
        binance_instrument_id: InstrumentId,
        backpack_instrument_id: InstrumentId,
        basis_threshold: float = 1.0,  # $1 threshold
        trade_size: float = 0.1,  # 0.1 ENA
    ):
        super().__init__()
        
        self.binance_id = binance_instrument_id
        self.backpack_id = backpack_instrument_id
        self.basis_threshold = basis_threshold
        self.trade_size = trade_size
        
        # Track positions
        self.in_position = False
        self.entry_basis = None
        
        # Statistics
        self.trades_executed = 0
        self.total_pnl = 0.0
        
    def on_start(self):
        """Called when strategy starts."""
        self.log.info("BasisTradingStrategy started")
        
        # Subscribe to quote ticks
        self.subscribe_quote_ticks(self.binance_id)
        self.subscribe_quote_ticks(self.backpack_id)
        
    def on_quote_tick(self, tick: QuoteTick):
        """Process incoming quote ticks."""
        
        # Update basis tracker (would be integrated in production)
        # For now, we'll demonstrate the logic
        
        # Get both quotes from cache
        binance_quotes = self.cache.quote_ticks(self.binance_id)
        backpack_quotes = self.cache.quote_ticks(self.backpack_id)
        
        if not (binance_quotes and backpack_quotes):
            return
        
        # Calculate basis
        binance_mid = float(binance_quotes[-1].bid_price + binance_quotes[-1].ask_price) / 2
        backpack_mid = float(backpack_quotes[-1].bid_price + backpack_quotes[-1].ask_price) / 2
        basis = backpack_mid - binance_mid
        
        # Trading logic
        if not self.in_position:
            self._check_entry(basis, binance_mid, backpack_mid)
        else:
            self._check_exit(basis)
    
    def _check_entry(self, basis: float, binance_price: float, backpack_price: float):
        """Check for entry conditions."""
        
        # Long basis: Backpack expensive, Binance cheap
        if basis > self.basis_threshold:
            self.log.info(f"Entry signal: Basis={basis:.2f} > {self.basis_threshold}")
            self.log.info("Action: Short Backpack, Long Binance")
            
            # In production, would place actual orders here
            # For demo, we track the trade
            self.in_position = True
            self.entry_basis = basis
            self.trades_executed += 1
            
        # Short basis: Binance expensive, Backpack cheap
        elif basis < -self.basis_threshold:
            self.log.info(f"Entry signal: Basis={basis:.2f} < -{self.basis_threshold}")
            self.log.info("Action: Long Backpack, Short Binance")
            
            self.in_position = True
            self.entry_basis = basis
            self.trades_executed += 1
    
    def _check_exit(self, basis: float):
        """Check for exit conditions."""
        
        # Exit when basis reverts to near zero
        if abs(basis) < 0.1:  # Within 10 cents of zero
            pnl = abs(self.entry_basis) - abs(basis)
            self.total_pnl += pnl
            
            self.log.info(f"Exit signal: Basis reverted to {basis:.2f}")
            self.log.info(f"Trade PnL: ${pnl:.2f}")
            
            self.in_position = False
            self.entry_basis = None
    
    def on_stop(self):
        """Called when strategy stops."""
        self.log.info(f"Strategy stopped. Total trades: {self.trades_executed}")
        self.log.info(f"Total PnL: ${self.total_pnl:.2f}")

print("✅ BasisTradingStrategy class defined")
print("\nStrategy features:")
print("  - Monitors basis spread in real-time")
print("  - Enters positions when basis exceeds threshold")
print("  - Exits when basis reverts to mean")
print("  - Tracks PnL and statistics")

## Part 7: Running the Complete System

Let's put everything together in a complete example that can be run.

In [ ]:
async def run_basis_monitor(duration_seconds: int = 60):
    """
    Run the complete basis trading system.
    
    Parameters
    ----------
    duration_seconds : int
        How long to run the monitor
    """
    
    print("Starting Basis Trading System...")
    print("="*60)
    
    try:
        # Setup clients
        binance_client, backpack_client = await setup_data_clients()
        
        # Subscribe to market data
        data_engine.subscribe_quote_ticks(binance_instrument_id)
        data_engine.subscribe_quote_ticks(backpack_instrument_id)
        
        print(f"\nMonitoring basis for {duration_seconds} seconds...")
        print("="*60)
        
        # Monitor for specified duration
        start_time = asyncio.get_event_loop().time()
        
        while (asyncio.get_event_loop().time() - start_time) < duration_seconds:
            # Get latest quotes
            binance_quotes = cache.quote_ticks(binance_instrument_id)
            backpack_quotes = cache.quote_ticks(backpack_instrument_id)
            
            # Update tracker
            if binance_quotes:
                basis_tracker.update_binance(binance_quotes[-1])
            if backpack_quotes:
                basis_tracker.update_backpack(backpack_quotes[-1])
            
            # Display current state
            current_basis = basis_tracker.get_current_basis()
            if current_basis is not None:
                stats = basis_tracker.get_statistics()
                print(f"\rBasis: ${current_basis:+.4f} | "
                      f"Range: [{stats.get('min_basis', 0):.4f}, "
                      f"{stats.get('max_basis', 0):.4f}] | "
                      f"Opportunities: {stats.get('opportunities', 0)}",
                      end="")
            
            await asyncio.sleep(1)
        
        print("\n\n" + "="*60)
        print("SESSION COMPLETE")
        print("="*60)
        
        # Display final statistics
        stats = basis_tracker.get_statistics()
        if stats:
            print(f"Data Points: {stats['data_points']}")
            print(f"Average Basis: ${stats['avg_basis']:.4f}")
            print(f"Basis Std Dev: ${stats['std_basis']:.4f}")
            print(f"Min Basis: ${stats['min_basis']:.4f}")
            print(f"Max Basis: ${stats['max_basis']:.4f}")
            print(f"Arbitrage Opportunities: {stats['opportunities']}")
        
        # Plot results
        basis_tracker.plot_basis()
        
    finally:
        # Cleanup
        print("\nShutting down...")
        data_engine.stop()
        await binance_client.disconnect()
        await backpack_client.disconnect()
        print("✅ Shutdown complete")

# Example of how to run (would be executed in async context)
print("To run the complete system:")
print("await run_basis_monitor(duration_seconds=60)")
print("\nOr from command line:")
print("python examples/live/basis_trading/ena_basis_example.py")

## Part 8: Risk Management and Production Considerations

### Risk Management

When implementing basis trading in production, consider:

1. **Position Limits**
   - Maximum position size per instrument
   - Maximum total exposure across all positions
   - Concentration limits

2. **Execution Risk**
   - Slippage on entry/exit
   - Partial fills
   - Network latency

3. **Market Risk**
   - Basis can widen before converging
   - Funding rates on perpetuals
   - Exchange-specific risks

4. **Operational Risk**
   - API rate limits
   - Connection stability
   - Data quality

In [ ]:
class RiskManager:
    """
    Risk management for basis trading.
    """
    
    def __init__(
        self,
        max_position_size: float = 1.0,  # Maximum position per side
        max_basis_exposure: float = 100.0,  # Maximum dollar exposure to basis
        stop_loss_basis: float = 5.0,  # Stop loss if basis moves against us
    ):
        self.max_position_size = max_position_size
        self.max_basis_exposure = max_basis_exposure
        self.stop_loss_basis = stop_loss_basis
        
        self.current_positions = {}
        self.total_exposure = 0.0
    
    def check_position_limit(self, size: float) -> bool:
        """Check if position size is within limits."""
        return size <= self.max_position_size
    
    def check_exposure_limit(self, new_exposure: float) -> bool:
        """Check if total exposure is within limits."""
        return (self.total_exposure + new_exposure) <= self.max_basis_exposure
    
    def should_stop_loss(self, entry_basis: float, current_basis: float) -> bool:
        """Check if position should be stopped out."""
        basis_move = abs(current_basis - entry_basis)
        return basis_move > self.stop_loss_basis
    
    def calculate_position_size(
        self,
        basis: float,
        volatility: float,
        account_balance: float,
    ) -> float:
        """Calculate optimal position size using Kelly criterion."""
        # Simplified Kelly sizing
        edge = abs(basis) / volatility  # Sharpe-like ratio
        kelly_fraction = min(edge / 4, 0.25)  # Cap at 25% of capital
        
        position_size = account_balance * kelly_fraction / 100  # Convert to base units
        
        # Apply position limits
        return min(position_size, self.max_position_size)

risk_manager = RiskManager()
print("✅ Risk Manager configured")
print(f"  - Max position size: {risk_manager.max_position_size} ENA")
print(f"  - Max basis exposure: ${risk_manager.max_basis_exposure}")
print(f"  - Stop loss trigger: ${risk_manager.stop_loss_basis} basis move")

## Summary and Next Steps

### What We've Learned

In this tutorial, we've covered:

1. **Basis Trading Concepts**
   - Understanding perpetual futures basis
   - Identifying arbitrage opportunities
   - Mean reversion strategies

2. **NautilusTrader Components**
   - MessageBus for event routing
   - Cache for data storage
   - DataEngine for market data management
   - Strategy framework for trading logic

3. **Multi-Exchange Integration**
   - Connecting to Binance and Backpack simultaneously
   - Normalizing data across exchanges
   - Handling different quote currencies

4. **Live Data Processing**
   - Subscribing to real-time quotes
   - Calculating spreads in real-time
   - Detecting trading opportunities

### Next Steps

To build on this foundation:

1. **Enhance the Strategy**
   - Add more sophisticated entry/exit logic
   - Implement dynamic position sizing
   - Include funding rate considerations

2. **Add Execution**
   - Implement actual order placement
   - Handle order management and fills
   - Add execution algorithms (TWAP, VWAP)

3. **Improve Risk Management**
   - Add portfolio-level risk limits
   - Implement correlation analysis
   - Add drawdown controls

4. **Backtesting**
   - Collect historical data
   - Backtest strategy performance
   - Optimize parameters

### Running the Complete Example

To run the complete example from the command line:

```bash
# Set your API keys in .env file
export BINANCE_API_KEY="your_binance_key"
export BINANCE_API_SECRET="your_binance_secret"
export BACKPACK_API_KEY="your_backpack_key"
export BACKPACK_API_SECRET="your_backpack_secret"

# Run the example
python examples/live/basis_trading/ena_basis_example.py
```

### Resources

- [NautilusTrader Documentation](https://nautilustrader.io/docs/)
- [Binance API Documentation](https://binance-docs.github.io/apidocs/)
- [Backpack API Documentation](https://docs.backpack.exchange/)
- [Basis Trading Strategies](https://www.cmegroup.com/education/courses/introduction-to-basis-trading.html)

### Disclaimer

This tutorial is for educational purposes only. Basis trading involves significant risks including:
- Market risk
- Execution risk
- Counterparty risk
- Technology risk

Always thoroughly test strategies in simulation before risking real capital.